In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [ ]:
!kaggle datasets download -d balabaskar/wonders-of-the-world-image-classification

Dataset URL: https://www.kaggle.com/datasets/balabaskar/wonders-of-the-world-image-classification
License(s): CC0-1.0
100% 432M/432M [00:03<00:00, 125MB/s]



In [ ]:
import zipfile
zip_ref=zipfile.ZipFile('/content/wonders-of-the-world-image-classification.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [ ]:
import pandas as pd
import numpy as np
import tensorflow
from tensorflow import keras
from keras.models import Sequential,Model
from keras.layers import Dense,BatchNormalization,Dropout,ReLU,Conv2D,MaxPooling2D,GlobalAveragePooling2D,Flatten
from keras.callbacks import EarlyStopping,ReduceLROnPlateau


In [ ]:
train_data=keras.utils.image_dataset_from_directory(
    directory='/content/Wonders of World/Wonders of World',
    labels="inferred",
    label_mode="int",
    batch_size=32,
    image_size=(256,256),
    subset="training",
    validation_split=0.2,
    seed=42

)

Found 3846 files belonging to 12 classes.
Using 3077 files for training.


In [ ]:
val_data=keras.utils.image_dataset_from_directory(
    directory="/content/Wonders of World/Wonders of World",
    labels="inferred",
    label_mode="int",
    batch_size=32,
    image_size=(256,256),
    subset='validation',
    validation_split=0.2,
    seed=42
)

Found 3846 files belonging to 12 classes.
Using 769 files for validation.


In [ ]:
3077+769

3846

In [ ]:
3077/3846*100

80.00520020800832

In [ ]:
def process(image,labels):
  image=tensorflow.cast(image/255,tensorflow.float32)
  return image,labels
train_data=train_data.map(process)
val_data=val_data.map(process)

In [ ]:
model=Sequential()

In [ ]:
model.add(Conv2D(64,kernel_size=(3,3),padding="valid",activation="relu",input_shape=(256,256,3)))
model.add(MaxPooling2D(pool_size=(2,2),padding='valid',strides=(2,2)))
model.add(BatchNormalization())
model.add(Conv2D(32,kernel_size=(3,3),padding="valid",activation="relu"))
model.add(MaxPooling2D(pool_size=(2,2),padding='valid'))
model.add(BatchNormalization())
model.add(Conv2D(12,kernel_size=(3,3),padding="valid",activation='relu'))
model.add(BatchNormalization())
model.add(GlobalAveragePooling2D())
model.add(Dense(12,activation="relu"))
model.add(Dropout(rate=0.3))
model.add(BatchNormalization())
model.add(Dense(32,activation='relu'))
model.add(BatchNormalization())
model.add(Dense(12,activation="softmax"))

In [ ]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_13 (Conv2D)              │ (None, 254, 254, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 127, 127, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 127, 127, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 125, 125, 32)   │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 62, 62, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 60, 60, 12)     │         3,468 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 60, 60, 12)     │            48 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 12)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 12)             │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 12)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 12)             │            48 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 32)             │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 12)             │           396 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,300 (98.83 KB)

 Trainable params: 24,996 (97.64 KB)

 Non-trainable params: 304 (1.19 KB)

In [ ]:
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [ ]:
history=model.fit(train_data,validation_data=(val_data),epochs=30,callbacks=EarlyStopping(patience=2,monitor='val_loss',verbose=2))

Epoch 1/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 25s 176ms/step - accuracy: 0.1290 - loss: 2.5877 - val_accuracy: 0.0533 - val_loss: 2.4884
Epoch 2/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 32s 133ms/step - accuracy: 0.2116 - loss: 2.2915 - val_accuracy: 0.1326 - val_loss: 2.4587
Epoch 3/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 20s 130ms/step - accuracy: 0.2535 - loss: 2.1516 - val_accuracy: 0.1027 - val_loss: 2.4546
Epoch 4/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 14s 146ms/step - accuracy: 0.2827 - loss: 2.0690 - val_accuracy: 0.2458 - val_loss: 2.1668
Epoch 5/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 15s 153ms/step - accuracy: 0.3039 - loss: 2.0187 - val_accuracy: 0.2367 - val_loss: 2.1354
Epoch 6/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 14s 139ms/step - accuracy: 0.3084 - loss: 1.9895 - val_accuracy: 0.3316 - val_loss: 2.0026
Epoch 7/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 13s 129ms/step - accuracy: 0.3302 - loss: 1.9349 - val_accuracy: 0.3017 - val_loss: 1.9790
Epoch 8/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 13s 134ms/step - accuracy: 0.3461 - loss: 1.9140 - val_accu

In [ ]:
from tensorflow.keras.applications.efficientnet import EfficientNetB7

In [ ]:
efficientnet=EfficientNetB7(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

258076736/258076736 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
efficientnet.trainable=False

In [ ]:
model=Sequential()

In [ ]:
model.add(efficientnet)
model.add(GlobalAveragePooling2D())
model.add(Dense(64,activation='relu'))
model.add(BatchNormalization())
model.add(Dense(12,activation='softmax'))

In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb7 (Functional)     │ (None, 7, 7, 2560)     │    64,097,687 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2560)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       163,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 12)             │           780 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 64,262,627 (245.14 MB)

 Trainable params: 164,812 (643.80 KB)

 Non-trainable params: 64,097,815 (244.51 MB)

In [ ]:
train_data=keras.utils.image_dataset_from_directory(
    directory='/content/Wonders of World/Wonders of World',
    labels="inferred",
    label_mode="int",
    batch_size=32,
    image_size=(224,224),
    subset="training",
    validation_split=0.2,
    seed=42

)

Found 3846 files belonging to 12 classes.
Using 3077 files for training.


In [ ]:
val_data=keras.utils.image_dataset_from_directory(
    directory='/content/Wonders of World/Wonders of World',
    labels="inferred",
    label_mode="int",
    batch_size=32,
    image_size=(224,224),
    subset="validation",
    validation_split=0.2,
    seed=42

)

Found 3846 files belonging to 12 classes.
Using 769 files for validation.


In [ ]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['accuracy'],optimizer='adam')

In [ ]:
history=model.fit(train_data,validation_data=(val_data),epochs=30,callbacks=EarlyStopping(patience=2,monitor='val_loss',verbose=2))

Epoch 1/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 192s 1s/step - accuracy: 0.8762 - loss: 0.4195 - val_accuracy: 0.9454 - val_loss: 0.3601
Epoch 2/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 29s 298ms/step - accuracy: 0.9760 - loss: 0.1171 - val_accuracy: 0.9792 - val_loss: 0.1535
Epoch 3/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 46s 348ms/step - accuracy: 0.9916 - loss: 0.0555 - val_accuracy: 0.9805 - val_loss: 0.0917
Epoch 4/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 35s 356ms/step - accuracy: 0.9909 - loss: 0.0429 - val_accuracy: 0.9844 - val_loss: 0.0628
Epoch 5/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 30s 306ms/step - accuracy: 0.9958 - loss: 0.0252 - val_accuracy: 0.9896 - val_loss: 0.0573
Epoch 6/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 30s 309ms/step - accuracy: 0.9958 - loss: 0.0200 - val_accuracy: 0.9896 - val_loss: 0.0604
Epoch 7/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 30s 308ms/step - accuracy: 0.9987 - loss: 0.0124 - val_accuracy: 0.9896 - val_loss: 0.0537
Epoch 8/30
97/97 ━━━━━━━━━━━━━━━━━━━━ 41s 310ms/step - accuracy: 0.9987 - loss: 0.0124 - val_accura